# HMDA 2024: kiểm tra độc lập bằng DuckDB (cài đặt thứ ba)

Hai pipeline Pandas và PySpark cùng đọc `rules.yaml`, nên việc chúng khớp nhau chỉ chứng tỏ hai cài đặt nhất quán,
chưa chứng tỏ luật được dịch đúng. Script này viết lại các phép tính chính bằng **DuckDB SQL**, với định nghĩa lấy
trực tiếp từ tài liệu FFIEC và Phụ lục A của báo cáo (gõ thẳng trong code, **không đọc rules.yaml**), rồi:

1. đếm số hồ sơ và tổng `loan_amount` theo nhiều lát cắt của toàn file để so với HMDA Data Browser;
2. tính lại các con số chính của mẫu nghiên cứu (lọc mẫu, kết quả theo loại vay, tỷ lệ từ chối theo bang và theo DTI,
   $O/E$ của từng bang và từng lender) và so với kết quả của pipeline Pandas.

Usage: `python independent_check.py [RAW_PARQUET] [PANDAS_RESULTS_DIR] [OUT_DIR]`
(mặc định tự tìm trong /kaggle/input hoặc thư mục HMDA_INPUT).

In [1]:
import glob
import json
import os
import sys
import time

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Kaggle thường có sẵn duckdb; nếu không thì cài
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb


def find(pattern, roots):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, "**", pattern), recursive=True))
        if hits:
            return hits[0]
    return None


ARGS = [a for a in sys.argv[1:] if not a.startswith("-") and not a.endswith(".json")]
ROOTS = [os.environ["HMDA_INPUT"]] if os.environ.get("HMDA_INPUT") else ["/kaggle/input", "."]
RAW = ARGS[0] if len(ARGS) > 0 else find("hmda_2024_raw.parquet", ROOTS)
PD_DIR = ARGS[1] if len(ARGS) > 1 else os.path.dirname(find("results/pandas/key_numbers.json", ROOTS) or "")
OUTROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("HMDA_OUT", "output")
OUT = ARGS[2] if len(ARGS) > 2 else f"{OUTROOT}/results/independent"
os.makedirs(OUT, exist_ok=True)
print("raw:", RAW, "\npandas results:", PD_DIR or "(không có)", "\nout:", OUT)

con = duckdb.connect()
con.execute(f"SET threads TO {os.cpu_count() or 2}")
con.execute(f"CREATE VIEW raw AS SELECT * FROM read_parquet('{RAW}')")
t0 = time.time()

raw: /kaggle/input/notebooks/study665/hmda-v2-convert/hmda_2024_raw.parquet 
pandas results: /kaggle/input/notebooks/study665/hmda-v2-pandas/results/pandas 
out: /kaggle/working/results/independent


## 1. Lát cắt của toàn file để so với HMDA Data Browser
Data Browser trả về số hồ sơ (`count`) và tổng `loan_amount` (`sum`) cho mỗi tổ hợp `action_taken` × một chiều.

In [2]:
DIMS = {"loan_types": "loan_type", "loan_purposes": "loan_purpose", "lien_statuses": "lien_status",
        "construction_methods": "construction_method", "states": "state_code"}
sl = []
for dim, col in DIMS.items():
    q = con.execute(f"""
        SELECT '{dim}' AS dimension, {col} AS value, action_taken, COUNT(*) AS count,
               SUM(TRY_CAST(loan_amount AS DOUBLE)) AS sum_loan_amount
        FROM raw GROUP BY ALL""").df()
    sl.append(q)
slices = pd.concat(sl, ignore_index=True).sort_values(["dimension", "value", "action_taken"])
slices.to_csv(f"{OUT}/db_slices.csv", index=False)
print("lát cắt:", len(slices), "dòng |", f"{time.time() - t0:.0f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lát cắt: 536 dòng | 6s


## 2. Mẫu nghiên cứu, viết lại từ định nghĩa
Core: hồ sơ (action_taken 1-5), thế chấp hạng nhất, closed-end, không vì mục đích kinh doanh, không reverse,
nhà site-built 1-4 căn, nhà ở chính. Decisioned: action_taken 1-3. Usable: DTI thuộc 19 giá trị công bố,
0 < CLTV <= 250, income > 0. Pricing: giải ngân, rate spread trong [-8, 15].

In [3]:
DTI_ORD = {"<20": 1, "20-<30": 2, "30-<36": 3, **{str(v): v - 32 for v in range(36, 50)}, "50-60": 18, ">60": 19}
dti_case = "CASE " + " ".join(f"WHEN d = '{k}' THEN {v}" for k, v in DTI_ORD.items()) + " END"
LTI_Q1, LTI_Q2 = 2.3550724637681157, 3.4036144578313254     # tam phân vị cố định (Phụ lục A)

con.execute(f"""
CREATE TABLE steps AS
SELECT row_id, lei, state_code, action_taken, loan_type, loan_purpose,
       (action_taken IN ('1','2','3','4','5')) AS s1,
       (lien_status = '1') AS s2, (open_end_line_of_credit = '2') AS s3,
       (business_or_commercial_purpose = '2') AS s4, (reverse_mortgage = '2') AS s5,
       (derived_dwelling_category = 'Single Family (1-4 Units):Site-Built') AS s6, (occupancy_type = '1') AS s7,
       debt_to_income_ratio, combined_loan_to_value_ratio, income, loan_amount, rate_spread,
       tract_to_msa_income_percentage
FROM raw""")
funnel = con.execute("""
SELECT COUNT(*) AS all_records, SUM(s1::INT) AS applications, SUM((s1 AND s2)::INT) AS first_lien,
       SUM((s1 AND s2 AND s3)::INT) AS closed_end, SUM((s1 AND s2 AND s3 AND s4)::INT) AS non_business,
       SUM((s1 AND s2 AND s3 AND s4 AND s5)::INT) AS non_reverse,
       SUM((s1 AND s2 AND s3 AND s4 AND s5 AND s6)::INT) AS site_built_1_4_unit,
       SUM((s1 AND s2 AND s3 AND s4 AND s5 AND s6 AND s7)::INT) AS principal_residence
FROM steps""").df().iloc[0].to_dict()

con.execute(f"""
CREATE TABLE core AS
SELECT row_id, lei, state_code, action_taken, loan_type, loan_purpose,
       action_taken IN ('1','2','3') AS decisioned, action_taken = '3' AS denied, action_taken = '1' AS orig,
       (SELECT {dti_case} FROM (SELECT replace(replace(debt_to_income_ratio, '%', ''), ' ', '') AS d)) AS dti,
       TRY_CAST(combined_loan_to_value_ratio AS DOUBLE) AS cltv, TRY_CAST(income AS DOUBLE) AS inc,
       TRY_CAST(loan_amount AS DOUBLE) AS amt, TRY_CAST(rate_spread AS DOUBLE) AS rs,
       NULLIF(TRY_CAST(tract_to_msa_income_percentage AS DOUBLE), 0) AS tract_pct
FROM steps WHERE s1 AND s2 AND s3 AND s4 AND s5 AND s6 AND s7""")
con.execute("""
CREATE TABLE usable AS
SELECT *, denied::INT AS y,
       CASE WHEN dti <= 2 THEN '<30' WHEN dti = 3 THEN '30-<36' WHEN dti <= 10 THEN '36-42'
            WHEN dti <= 17 THEN '43-49' WHEN dti = 18 THEN '50-60' ELSE '>60' END AS db,
       CASE WHEN cltv <= 80 THEN '<=80' WHEN cltv <= 90 THEN '80-90' WHEN cltv <= 95 THEN '90-95'
            WHEN cltv <= 100 THEN '95-100' ELSE '>100' END AS cb,
       amt / (1000 * inc) AS lti
FROM core WHERE decisioned AND dti IS NOT NULL AND cltv > 0 AND cltv <= 250 AND inc > 0""")
con.execute(f"""ALTER TABLE usable ADD COLUMN lb VARCHAR;
UPDATE usable SET lb = CASE WHEN lti <= {LTI_Q1} THEN 'T1' WHEN lti <= {LTI_Q2} THEN 'T2' ELSE 'T3' END""")
counts = con.execute("""
SELECT (SELECT SUM(decisioned::INT) FROM core) AS decisioned, (SELECT COUNT(*) FROM usable) AS usable,
       (SELECT COUNT(*) FROM core WHERE orig AND rs BETWEEN -8 AND 15) AS pricing""").df().iloc[0].to_dict()
funnel.update(counts)
print("đếm mẫu:", {k: int(v) for k, v in funnel.items()}, f"| {time.time() - t0:.0f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

đếm mẫu: {'all_records': 12259199, 'applications': 10781992, 'first_lien': 7981147, 'closed_end': 7258042, 'non_business': 6694328, 'non_reverse': 6690389, 'site_built_1_4_unit': 6088849, 'principal_residence': 5775325, 'decisioned': 4514505, 'usable': 4100051, 'pricing': 3616549} | 31s


## 3. Các bảng để so với Pandas

In [4]:
TYPE = "CASE loan_type WHEN '1' THEN 'conventional' WHEN '2' THEN 'FHA' WHEN '3' THEN 'VA' WHEN '4' THEN 'USDA' END"
T = {}
T["outcomes_by_type"] = con.execute(f"""
SELECT {TYPE} AS type, COUNT(*) AS n_applications, AVG((action_taken='1')::INT) AS originated,
       AVG((action_taken='3')::INT) AS denied, AVG((action_taken='4')::INT) AS withdrawn
FROM core GROUP BY 1""").df()
T["loan_originated_by_type"] = con.execute(f"""
SELECT {TYPE} AS type, COUNT(amt) AS n, AVG(amt) AS mean, quantile_cont(amt, 0.5) AS median
FROM core WHERE orig GROUP BY 1""").df()
T["rate_spread_by_type"] = con.execute(f"""
SELECT {TYPE} AS type, COUNT(*) AS n, AVG(rs) AS mean, quantile_cont(rs, 0.5) AS median
FROM core WHERE orig AND rs BETWEEN -8 AND 15 GROUP BY 1""").df()
T["denial_by_state"] = con.execute("""
SELECT state_code, COUNT(*) AS n_decisioned, SUM(denied::INT) AS n_denied FROM core WHERE decisioned GROUP BY 1""").df()
T["denial_by_dti_19"] = con.execute("""
SELECT dti AS dti_ordinal, COUNT(*) AS n, AVG(y) AS denial_rate FROM usable GROUP BY 1""").df()
T["denial_type_x_purpose"] = con.execute(f"""
SELECT {TYPE} AS type, loan_purpose, COUNT(*) AS n_decisioned, SUM(denied::INT) AS n_denied
FROM core WHERE decisioned GROUP BY ALL""").df()

# xác suất kỳ vọng: tỷ lệ quốc gia của ô mở rộng (ô rủi ro × loại vay × mục đích), ô < 30 hồ sơ dùng ô rủi ro
con.execute("""
CREATE TABLE pe AS
WITH ext AS (SELECT db, cb, lb, loan_type, loan_purpose, COUNT(*) AS n_e, AVG(y) AS r_e FROM usable GROUP BY ALL),
     rc AS (SELECT db, cb, lb, AVG(y) AS r_c FROM usable GROUP BY ALL)
SELECT u.*, CASE WHEN ext.n_e >= 30 THEN ext.r_e ELSE rc.r_c END AS p
FROM usable u JOIN ext USING (db, cb, lb, loan_type, loan_purpose) JOIN rc USING (db, cb, lb)""")
T["oe_state"] = con.execute("SELECT state_code, COUNT(*) AS n, SUM(y) AS O, SUM(p) AS E FROM pe GROUP BY 1").df()
T["oe_lender"] = con.execute("SELECT lei, COUNT(*) AS n, SUM(y) AS O, SUM(p) AS E FROM pe GROUP BY 1").df()
T["oe_national"] = con.execute("SELECT SUM(y) / SUM(p) AS oe FROM pe").df()
# số liệu để quy đổi kết quả ra đô la trong phần diễn giải tài chính
MAG = con.execute("""
SELECT (SELECT quantile_cont(amt, 0.5) FROM core WHERE orig AND rs BETWEEN -8 AND 15 AND loan_type = '1' AND loan_purpose = '32')
           AS median_loan_conv_cashout,
       (SELECT quantile_cont(amt, 0.5) FROM core WHERE orig AND rs BETWEEN -8 AND 15 AND loan_type = '1' AND loan_purpose = '1')
           AS median_loan_conv_purchase,
       (SELECT quantile_cont(inc, 0.5) FROM usable WHERE loan_type = '1' AND dti = 17) AS median_income_conv_dti49,
       (SELECT COUNT(*) FROM usable WHERE loan_type = '1' AND dti = 17) AS n_conv_dti49""").df().iloc[0].to_dict()
json.dump({k: float(v) for k, v in MAG.items()}, open(f"{OUT}/ind_magnitudes.json", "w"), indent=1)
print("quy đổi:", MAG)
for k, v in T.items():
    v.to_csv(f"{OUT}/ind_{k}.csv", index=False)
json.dump({k: int(v) for k, v in funnel.items()}, open(f"{OUT}/ind_funnel.json", "w"), indent=1)
print("xong các bảng |", f"{time.time() - t0:.0f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

quy đổi: {'median_loan_conv_cashout': 215000.0, 'median_loan_conv_purchase': 335000.0, 'median_income_conv_dti49': 97.0, 'n_conv_dti49': 183248.0}
xong các bảng | 34s


## 4. So với kết quả của pipeline Pandas
Đếm phải khớp tuyệt đối; trung bình, trung vị, tỷ lệ và $O/E$ so theo sai lệch tương đối $\le 10^{-9}$.

In [5]:
rows = []


def add(quantity, key, ind, pdv, exact):
    ind, pdv = float(ind), float(pdv)
    d = 0.0 if ind == pdv else abs(ind - pdv) / max(abs(ind), abs(pdv))
    rows.append({"quantity": quantity, "key": key, "independent": ind, "pandas": pdv, "rel_diff": d,
                 "agree": (ind == pdv) if exact else d <= 1e-9})


if PD_DIR and os.path.exists(os.path.join(PD_DIR, "eq_1_filter_counts.csv")):
    P = lambda n: pd.read_csv(os.path.join(PD_DIR, f"{n}.csv"), keep_default_na=False, na_values=[""])
    f1 = P("eq_1_filter_counts").set_index("step")["rows"]
    for k, v in funnel.items():
        if k in f1.index:
            add("sample sizes", k, v, f1[k], True)
    a = T["outcomes_by_type"].set_index("type"); b = P("sq1_outcomes_by_type").set_index("type")
    for t in a.index:
        add("applications by loan type", t, a.at[t, "n_applications"], b.at[t, "n_applications"], True)
        for col in ["originated", "denied", "withdrawn"]:
            add(f"share {col} by loan type", t, a.at[t, col], b.at[t, col], False)
    for name, pdname in [("loan_originated_by_type", "sq1_loan_amount_originated_by_type"), ("rate_spread_by_type", "sq2_rate_spread_by_type")]:
        a = T[name].set_index("type"); b = P(pdname).set_index("type")
        for t in a.index:
            add(f"{name}: n", t, a.at[t, "n"], b.at[t, "n"], True)
            add(f"{name}: mean", t, a.at[t, "mean"], b.at[t, "mean"], False)
            add(f"{name}: median", t, a.at[t, "median"], b.at[t, "median"], False)
    a = T["denial_by_state"].set_index("state_code"); b = P("eq_5_denial_by_state").set_index("state_code")
    for s in a.index:
        add("decisioned by state", s, a.at[s, "n_decisioned"], b.at[s, "n_decisioned"], True)
        add("denied by state", s, a.at[s, "n_denied"], b.at[s, "n_denied"], True)
    a = T["denial_by_dti_19"].set_index("dti_ordinal"); b = P("sq2_denial_by_dti_19"); b = b.set_index(b["dti_ordinal"].astype(int))
    for k in a.index:
        add("usable by DTI value", int(k), a.at[k, "n"], b.at[int(k), "n"], True)
        add("denial rate by DTI value", int(k), a.at[k, "denial_rate"], b.at[int(k), "denial_rate"], False)
    for name, pdname, key in [("oe_state", "sq3_state_oe", "state_code"), ("oe_lender", "sq4_lender_oe", "lei")]:
        a = T[name].set_index(key); b = P(pdname).set_index(key)
        for k in b.index:
            if k in a.index:
                add(f"{name}: n", k, a.at[k, "n"], b.at[k, "n"], True)
                add(f"{name}: O", k, a.at[k, "O"], b.at[k, "O"], True)
                add(f"{name}: E", k, a.at[k, "E"], b.at[k, "E"], False)
    kn = json.load(open(os.path.join(PD_DIR, "key_numbers.json")))
    if "national_oe" in kn:
        add("national O/E", "all", T["oe_national"]["oe"].iloc[0], kn["national_oe"], False)
    R = pd.DataFrame(rows)
    R.to_csv(f"{OUT}/independent_vs_pandas.csv", index=False)
    S = R.groupby("quantity", sort=False).agg(values=("agree", "size"), agree=("agree", "all"), max_rel_diff=("rel_diff", "max")).reset_index()
    S.to_csv(f"{OUT}/independent_vs_pandas_summary.csv", index=False)
    pd.set_option("display.width", 200)
    print(S.to_string(index=False))
    print(f"\n{len(R)} giá trị; {int((~R['agree']).sum())} lệch;", "Tất cả khớp" if R["agree"].all() else "CÓ LỆCH")
    if not R["agree"].all():
        print(R[~R["agree"]].head(20).to_string(index=False))
print(f"tổng thời gian {time.time() - t0:.0f}s")

                       quantity  values  agree  max_rel_diff
                   sample sizes      11   True  0.000000e+00
      applications by loan type       4   True  0.000000e+00
  share originated by loan type       4   True  0.000000e+00
      share denied by loan type       4   True  9.670941e-16
   share withdrawn by loan type       4   True  4.888851e-16
     loan_originated_by_type: n       4   True  0.000000e+00
  loan_originated_by_type: mean       4   True  1.487295e-16
loan_originated_by_type: median       4   True  0.000000e+00
         rate_spread_by_type: n       4   True  0.000000e+00
      rate_spread_by_type: mean       4   True  6.666271e-15
    rate_spread_by_type: median       4   True  0.000000e+00
            decisioned by state      55   True  0.000000e+00
                denied by state      55   True  0.000000e+00
            usable by DTI value      19   True  0.000000e+00
       denial rate by DTI value      19   True  9.610730e-16
                    oe_s

---
# Phần 2: `pred_subgroups.py`

# HMDA 2024: đánh giá mô hình dự báo theo từng nhóm hồ sơ

Dùng dự đoán out-of-fold mà notebook Pandas ghi ra (`pred_oof.parquet`, fold theo lender) và kiểm tra xem mô hình
có dự đoán đều tay giữa các nhóm hay không: loại vay, mục đích vay, tract LMI và băng DTI. Với mỗi nhóm: số hồ sơ,
tỷ lệ từ chối thực tế, trung bình dự đoán (CITL = dự đoán - thực tế), ROC-AUC và Brier.
Các đặc điểm nhân khẩu học không dùng ở đây (xem hạn chế (vii) của báo cáo).

Usage: `python pred_subgroups.py [PRED_OOF_PARQUET] [RAW_PARQUET] [OUT_DIR]`

In [6]:
import glob
import os
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

try:
    import duckdb
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb


def find(pattern, roots):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, "**", pattern), recursive=True))
        if hits:
            return hits[0]
    return None


ARGS = [a for a in sys.argv[1:] if not a.startswith("-") and not a.endswith(".json")]
ROOTS = [os.environ["HMDA_INPUT"]] if os.environ.get("HMDA_INPUT") else ["/kaggle/input", "."]
OOF = ARGS[0] if len(ARGS) > 0 else find("pred_oof.parquet", ROOTS)
RAW = ARGS[1] if len(ARGS) > 1 else find("hmda_2024_raw.parquet", ROOTS)
OUTROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("HMDA_OUT", "output")
OUT = ARGS[2] if len(ARGS) > 2 else f"{OUTROOT}/results/predictive"
os.makedirs(OUT, exist_ok=True)
print("oof:", OOF, "\nraw:", RAW)

oof: /kaggle/input/notebooks/study665/hmda-v2-pandas/results/pandas/pred_oof.parquet 
raw: /kaggle/input/notebooks/study665/hmda-v2-convert/hmda_2024_raw.parquet


In [7]:
con = duckdb.connect()
d = con.execute(f"""
SELECT o.y, o.p_lr, o.p_gbt,
       CASE r.loan_type WHEN '1' THEN 'conventional' WHEN '2' THEN 'FHA' WHEN '3' THEN 'VA' WHEN '4' THEN 'USDA' END AS loan_type,
       CASE r.loan_purpose WHEN '1' THEN 'purchase' WHEN '2' THEN 'home improvement' WHEN '31' THEN 'refinance'
            WHEN '32' THEN 'cash-out refinance' WHEN '4' THEN 'other' ELSE 'not applicable' END AS purpose,
       CASE WHEN NULLIF(TRY_CAST(r.tract_to_msa_income_percentage AS DOUBLE), 0) IS NULL THEN 'tract unknown'
            WHEN TRY_CAST(r.tract_to_msa_income_percentage AS DOUBLE) < 80 THEN 'LMI tract' ELSE 'other tract' END AS tract,
       CASE replace(replace(r.debt_to_income_ratio, '%', ''), ' ', '')
            WHEN '<20' THEN '<30' WHEN '20-<30' THEN '<30' WHEN '30-<36' THEN '30-<36'
            WHEN '50-60' THEN '50-60' WHEN '>60' THEN '>60'
            ELSE CASE WHEN TRY_CAST(replace(r.debt_to_income_ratio, '%', '') AS INT) <= 42 THEN '36-42' ELSE '43-49' END
       END AS dti_band
FROM read_parquet('{OOF}') o JOIN read_parquet('{RAW}') r USING (row_id)""").df()
print(len(d), "hồ sơ")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

4100051 hồ sơ


In [8]:
rows = []
for dim in ["all", "loan_type", "purpose", "tract", "dti_band"]:
    groups = [("all", d)] if dim == "all" else list(d.groupby(dim))
    for g, x in groups:
        r = {"dimension": dim, "group": g, "n": len(x), "observed": x["y"].mean()}
        for m in ["lr", "gbt"]:
            p = x[f"p_{m}"]
            r[f"{m}_predicted"] = p.mean()
            r[f"{m}_citl"] = p.mean() - x["y"].mean()
            r[f"{m}_brier"] = float(np.mean((p - x["y"]) ** 2))
            r[f"{m}_auc"] = roc_auc_score(x["y"], p) if x["y"].nunique() == 2 else np.nan
        rows.append(r)
S = pd.DataFrame(rows)
S.to_csv(f"{OUT}/pred_subgroups.csv", index=False)
pd.set_option("display.width", 220)
print(S.round(4).to_string(index=False))

dimension              group       n  observed  lr_predicted  lr_citl  lr_brier  lr_auc  gbt_predicted  gbt_citl  gbt_brier  gbt_auc
      all                all 4100051    0.1401        0.1410   0.0009    0.0886  0.8047         0.1403    0.0002     0.0795   0.8356
loan_type                FHA  935743    0.1991        0.2004   0.0012    0.1204  0.7769         0.1992    0.0000     0.1077   0.8136
loan_type               USDA   35469    0.1151        0.1156   0.0005    0.0883  0.6840         0.1154    0.0003     0.0734   0.7775
loan_type                 VA  383349    0.1479        0.1491   0.0012    0.1061  0.7881         0.1471   -0.0008     0.0928   0.8058
loan_type       conventional 2745490    0.1192        0.1199   0.0008    0.0754  0.8083         0.1196    0.0004     0.0681   0.8360
  purpose cash-out refinance  655837    0.3098        0.3125   0.0027    0.1773  0.6960         0.3115    0.0017     0.1578   0.7538
  purpose   home improvement   52538    0.4084        0.4042  -0.0043

In [9]:
# In các file kết quả (nhỏ) kèm md5 để lưu lại vào repo
import glob, hashlib, os
for p in sorted(glob.glob("/kaggle/working/results/**/*", recursive=True)):
    if os.path.isfile(p) and os.path.getsize(p) < 200_000 and not p.endswith((".parquet", ".png", ".npz")):
        b = open(p, "rb").read()
        print("@@@@ FILE", os.path.relpath(p, "/kaggle/working"), hashlib.md5(b).hexdigest(), len(b))
        print(b.decode("utf-8"))
        print("@@@@ END")

@@@@ FILE results/independent/db_slices.csv 339fb9b55836d24b403c8e47eda99344 16854
dimension,value,action_taken,count,sum_loan_amount
construction_methods,1,1,6015448,2011577460000.0
construction_methods,1,2,315507,89325295000.0
construction_methods,1,3,1831090,410729200000.0
construction_methods,1,4,1487234,527028850000.0
construction_methods,1,5,479544,121603790000.0
construction_methods,1,6,1244072,529704330000.0
construction_methods,1,7,46055,14975485000.0
construction_methods,1,8,151826,62814920000.0
construction_methods,2,1,181637,29642475000.0
construction_methods,2,2,45725,5823245000.0
construction_methods,2,3,272372,33944210000.0
construction_methods,2,4,51039,9194005000.0
construction_methods,2,5,102396,12966580000.0
construction_methods,2,6,32432,5617780000.0
construction_methods,2,7,1257,193505000.0
construction_methods,2,8,1565,291395000.0
lien_statuses,1,1,4770735,1892955125000.0
lien_statuses,1,2,260592,85901090000.0
lien_statuses,1,3,1246459,352983825000.0
lien_statuses